# MC-FPET pilot on Kaggle
Runs the configs in `RUNS` (default: all 5 — adaptformer, fpet-r2, mcpet-r2, fpet-r4, mcpet-r4) for every dataset in `DATASETS` and seed in `SEEDS` (set them in cell 0).
Finished runs are skipped, so the notebook can be re-run after a timeout.

Settings: GPU T4 x2, Internet on, input dataset `fpet-assets` (must contain each dataset as a folder or as `<name>.zip`).
To resume after a timeout, also attach the previous version's output as an input: cell 3 restores its finished runs.

Rough time on a T4: dsprites_loc fpet ~2 h, mcpet ~3 h with the EXTRA flags; mcpet-r4 smallnorb_azi ~2 h; all 5 runs per seed: cifar ~6 h, dtd ~5 h, smallnorb_azi ~8 h, dsprites_loc ~10 h. Kaggle stops a session at 12 h.


In [ ]:
# 0. what to run: MC-FPET (mcpet) jobs only, as (dataset, run, seed)
JOBS = [
    ("smallnorb_azi", "mcpet-r4", 2),
]
assert all(run.startswith("mcpet-") for _, run, _ in JOBS), "this notebook runs mcpet only"
DATASETS = sorted({d for d, _, _ in JOBS})

# dsprites_loc has 73,728 test images, so evaluating every 10 epochs (and the full branch for MC)
# pushes MC runs past Kaggle's 12 h limit. Evaluate at epochs 24/49/74/99 and skip the full-branch eval.
EXTRA = {"dsprites_loc": "--eval_every 25 --no_full_eval"}


In [ ]:
# 1. code
import os
if not os.path.isdir("/kaggle/working/kims_fpet"):
    !git clone -b merge-consistency https://github.com/kelvenniky/kims_fpet.git /kaggle/working/kims_fpet
%cd /kaggle/working/kims_fpet
!pip install -q timm==0.4.12

In [ ]:
# 2. weights + data
import os, glob
# works for any account and any nesting (Kaggle nests uploads as name/name)
A = sorted(glob.glob("/kaggle/input/**/fpet-assets", recursive=True), key=len)[0]
print("assets:", A)

# ViT-B_16.npz may itself be a folder holding the real file, so pick the actual file
if not os.path.isfile("ViT-B_16.npz"):
    if os.path.lexists("ViT-B_16.npz"):
        os.remove("ViT-B_16.npz")
    files = [p for p in glob.glob(f"{A}/**/ViT-B_16.npz", recursive=True) if os.path.isfile(p)]
    if files:
        os.symlink(files[0], "ViT-B_16.npz")
    else:
        !wget -q https://storage.googleapis.com/vit_models/imagenet21k/ViT-B_16.npz
!ls -laL ViT-B_16.npz

# copy each dataset to local disk: reading thousands of small images from /kaggle/input is slow
# (dsprites_loc's first eval took ~30 min from there). /tmp is not part of the saved output.
# uploaded folders can be nested (dtd/dtd), so find the real folder first
import shutil
V = "/tmp/vtab"
os.makedirs(V, exist_ok=True)
for name in DATASETS:
    dst = f"{V}/{name}"
    if not os.path.isfile(f"{dst}/train800val200.txt"):
        shutil.rmtree(dst, ignore_errors=True)
        hits = glob.glob(f"{A}/**/{name}/train800val200.txt", recursive=True)
        zips = [p for p in glob.glob(f"{A}/**/{name}.zip", recursive=True) if os.path.isfile(p)]
        if hits:
            !cp -r "{os.path.dirname(hits[0])}" "{dst}"
        elif zips:
            # unzip, then move the folder holding train800val200.txt to dst (zips may nest name/name)
            tmp = f"{V}/_unzip_{name}"
            shutil.rmtree(tmp, ignore_errors=True)
            !unzip -q "{zips[0]}" -d "{tmp}"
            inner = glob.glob(f"{tmp}/**/train800val200.txt", recursive=True)
            assert inner, f"{zips[0]} has no train800val200.txt"
            shutil.move(os.path.dirname(inner[0]), dst)
            shutil.rmtree(tmp, ignore_errors=True)
        else:
            raise FileNotFoundError(f"{name} not found in {A}: add the folder or {name}.zip to fpet-assets")
    n_test = len(os.listdir(f"{dst}/images/test"))
    print(name, "->", dst, f"({n_test} test images)")
os.environ["VTAB_ROOT"] = V


In [ ]:
# 3. restore only the mcpet jobs above from any attached previous output (for resuming after a timeout)
import shutil
for s in glob.glob("/kaggle/input/**/pilot/*/seed*/*/summary.json", recursive=True):
    run, seed, dataset = s.split("/")[-4:-1]
    if (dataset, run, int(seed[4:])) not in JOBS:
        continue
    dst = f"result/pilot/{run}/{seed}/{dataset}"
    if not os.path.exists(f"{dst}/summary.json"):
        shutil.copytree(os.path.dirname(s), dst, dirs_exist_ok=True)
        print("restored", run, seed, dataset)
!find result -name summary.json 2>/dev/null | sort


In [ ]:
# 4. run the mcpet jobs (finished ones are skipped); results are zipped after every run
%cd /kaggle/working/kims_fpet
FLAGS = {"mcpet-r2": "--r_layer 2 --consistency", "mcpet-r4": "--r_layer 4 --consistency"}
for dataset, run, seed in JOBS:
    d = f"result/pilot/{run}/seed{seed}"
    if os.path.isfile(f"{d}/{dataset}/summary.json"):
        print("skip", run, seed, dataset, "(done)")
        continue
    os.makedirs(f"{d}/{dataset}", exist_ok=True)
    !CUDA_VISIBLE_DEVICES=0 python main.py --dataset {dataset} --method adaptformer --dim 8 --bit 32 --load_config --seed {seed} --model_path {d} {FLAGS[run]} {EXTRA.get(dataset, '')} 2>&1 | tee {d}/{dataset}/stdout.log
    !zip -qr /kaggle/working/pilot_results.zip result/pilot
    !cat {d}/{dataset}/summary.json
!find result -name summary.json | sort
